# CryptoGraph — LSTM: will a coin shock in the next hour?

An **LSTM** reads one coin's last 168 hours and predicts whether that coin has a price shock in the next
hour (a move larger than ~2.3× its recent volatility). It uses no graph, so it is our "does the network help?"
baseline. Metric: **AUPRC** (random guessing ≈ 0.04 here). 5 seeds. About 20–40 minutes on a T4.

**Run:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. Needs `cryptograph_data_v2.zip` in `MyDrive/CryptoGraph`.

## 1. Load the data from Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/CryptoGraph'
!unzip -q -o {DRIVE}/cryptograph_data_v2.zip 'data/processed/v2/1h/panel.pt' 'data/processed/v2/1h/lstm_index_*' -d /content
D = '/content/data/processed/v2/1h'
!ls -lh $D

In [ ]:
import os, copy, time
import torch
import numpy as np, pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score

DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEV, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

def set_seed(seed):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def focal_loss(logits, target, mask, gamma=2.0, alpha=0.25):
    # focal loss for rare positives (Lin et al. 2017); only entries where mask is True count
    t = target.float()
    bce = F.binary_cross_entropy_with_logits(logits, t, reduction='none')
    p = torch.sigmoid(logits)
    p_t = p * t + (1 - p) * (1 - t)
    loss = (alpha * t + (1 - alpha) * (1 - t)) * (1 - p_t) ** gamma * bce
    m = mask.float()
    return (loss * m).sum() / m.sum().clamp_min(1.0)

def auprc(y, s):
    return float(average_precision_score(y, s)) if y.sum() > 0 else float('nan')

def auroc(y, s):
    return float(roc_auc_score(y, s)) if 0 < y.sum() < len(y) else float('nan')

SAVE = f'{DRIVE}/results_lstm_shock'
os.makedirs(SAVE, exist_ok=True)

## 2. Inputs: 168-hour windows of one coin
Each sample is (hour *t*, coin *n*). The model sees that coin's 6 features over the last 168 hours; nothing after hour *t* is used.

In [ ]:
panel = torch.load(f'{D}/panel.pt', weights_only=False)
X = torch.nan_to_num(panel['X'].float()).to(DEV)            # (hours, coins, features): scaled inputs, NaN -> 0
Y_SHOCK = panel['y_shock'].float().to(DEV)                  # (hours, coins): shock in the next hour
Y_RV = panel['y_rv'].float().to(DEV)                        # (hours, coins, 3): realized volatility next 1/6/24 h
INDEX = {s: torch.load(f'{D}/lstm_index_{s}.pt', weights_only=False)['index'].to(DEV) for s in ['train', 'val', 'test']}
L = 168                                                     # input window: last 168 hours of one coin
print('features:', panel['feature_names'])
print({s: len(v) for s, v in INDEX.items()}, 'samples (train / val / test)')

def get_batch(idx):
    # idx: (B, 2) rows of (hour t, coin n) -> inputs X[t-167..t, n] of shape (B, 168, F)
    t, n = idx[:, 0], idx[:, 1]
    hours = t[:, None] - torch.arange(L - 1, -1, -1, device=DEV)[None, :]
    return X[hours, n[:, None]]

## 3. Model

In [ ]:
class LSTMModel(nn.Module):
    # same as cryptograph.models.lstm.CryptoLSTM with hidden 32, 1 layer, dropout 0.2 (configs/model/lstm.yaml)
    def __init__(self, n_features, n_out, hidden=32, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden // 2, n_out))

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1])

TARGET, N_OUT, KEY = Y_SHOCK, 1, 'auprc'               # early stopping keeps the epoch with the best validation AUPRC
LOSS = lambda pred, y: focal_loss(pred.squeeze(-1), y, torch.ones_like(y, dtype=torch.bool))

## 4. Training
Up to 30 epochs; training stops after 5 epochs without improvement on the validation period (2022–23) and keeps the best epoch. The test period (2024–26) is scored once at the end.

In [ ]:
EPOCHS, PATIENCE, BATCH, LR, WD = 30, 5, 512, 1e-3, 1e-4      # same as configs/experiment/lstm_1h_shock_final.yaml

def predict(model, split):
    model.eval(); preds, ys = [], []
    with torch.no_grad():
        for i in range(0, len(INDEX[split]), 4096):
            idx = INDEX[split][i:i + 4096]
            preds.append(model(get_batch(idx)))
            ys.append(TARGET[idx[:, 0], idx[:, 1]])
    return torch.cat(preds).cpu(), torch.cat(ys).cpu()

def score(pred, y):
    y, s = y.numpy(), pred.squeeze(-1).numpy()
    return {'auprc': auprc(y, s), 'auroc': auroc(y, s), 'positive_rate': float(y.mean())}

def train_one_seed(seed):
    set_seed(seed)
    model = LSTMModel(X.shape[-1], N_OUT).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    g = torch.Generator(device='cpu').manual_seed(seed)
    best, best_score, bad, history = None, -np.inf, 0, []
    for epoch in range(EPOCHS):
        model.train(); t0 = time.time(); losses = []
        for b in torch.randperm(len(INDEX['train']), generator=g).split(BATCH):
            idx = INDEX['train'][b.to(DEV)]
            y = TARGET[idx[:, 0], idx[:, 1]]
            loss = LOSS(model(get_batch(idx)), y)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            losses.append(loss.item())
        val = score(*predict(model, 'val'))
        history.append({'epoch': epoch, 'train_loss': np.mean(losses), **{'val_' + k: v for k, v in val.items()}})
        print(f"seed {seed} epoch {epoch:2d}  train loss {np.mean(losses):.5f}  val {KEY} {val[KEY]:.5g}  ({time.time() - t0:.0f}s)")
        if val[KEY] > best_score:
            best, best_score, bad, best_epoch = copy.deepcopy(model.state_dict()), val[KEY], 0, epoch
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f'early stop (best epoch {best_epoch})'); break
    model.load_state_dict(best)
    result = {'seed': seed, 'best_epoch': best_epoch}
    for split in ['val', 'test']:                      # test is scored once, with the model chosen on validation
        result.update({f'{split}_{k}': v for k, v in score(*predict(model, split)).items()})
    torch.save({'model_state': best, 'result': result, 'history': history}, f'{SAVE}/lstm_shock_seed{seed}.pt')
    return result

## 5. Train 5 seeds and save the models
Each trained model is saved to `MyDrive/CryptoGraph/results_lstm_shock/` as soon as it finishes.

In [ ]:
results = []
for seed in range(5):
    results.append(train_one_seed(seed))
    pd.DataFrame(results).to_csv(f'{SAVE}/results.csv', index=False)
res = pd.DataFrame(results)
display(res.round(5))

## 6. Summary (mean ± std over the 5 seeds)

In [ ]:
display(res.drop(columns=['seed']).agg(['mean', 'std']).T.round(5))
!cd $DRIVE && zip -q -r results_lstm_shock.zip results_lstm_shock
print('Saved. Send back: MyDrive/CryptoGraph/results_lstm_shock.zip')